대분류 컬럼

In [ ]:
df['main_category'] = df['category_code'].str.split('.').str[0]

In [ ]:
event_first_time_main = (
    df
    .groupby([
        'main_category',
        'user_session',
        'product_id',
        'event_type'
    ])['event_time']
    .min()
    .unstack()
    .reset_index()
)

In [ ]:
for col in ['view', 'cart', 'purchase']:
    if col not in event_first_time_main.columns:
        event_first_time_main[col] = pd.NaT

# 존재 여부
event_first_time_main['has_view'] = event_first_time_main['view'].notna()
event_first_time_main['has_cart'] = event_first_time_main['cart'].notna()
event_first_time_main['has_purchase'] = event_first_time_main['purchase'].notna()

# 순차 퍼널
event_first_time_main['view_to_cart'] = (
    event_first_time_main['has_view'] &
    event_first_time_main['has_cart'] &
    (event_first_time_main['view'] < event_first_time_main['cart'])
)

event_first_time_main['view_to_cart_to_purchase'] = (
    event_first_time_main['has_view'] &
    event_first_time_main['has_cart'] &
    event_first_time_main['has_purchase'] &
    (event_first_time_main['view'] < event_first_time_main['cart']) &
    (event_first_time_main['cart'] < event_first_time_main['purchase'])
)

In [ ]:
funnel_df = event_first_time_main[[
    'main_category',
    'user_session',
    'product_id',
    'has_view',
    'view_to_cart',
    'view_to_cart_to_purchase'
]].rename(columns={
    'has_view': 'view'
})

In [ ]:
category_funnel = (
    funnel_df.groupby('main_category')
    .agg(
        view=('view', 'sum'),
        view_to_cart=('view_to_cart', 'sum'),
        conversion=('view_to_cart_to_purchase', 'sum')
    )
    .reset_index()
)

In [ ]:
category_funnel['view_to_cart_rate'] = (
    category_funnel['view_to_cart'] /
    category_funnel['view'] * 100
)

category_funnel['cart_to_purchase_rate'] = (
    category_funnel['conversion'] /
    category_funnel['view_to_cart'] * 100
)

category_funnel['total_conversion_rate'] = (
    category_funnel['conversion'] /
    category_funnel['view'] * 100
)

In [ ]:
category_funnel = category_funnel.sort_values(
    'total_conversion_rate',
    ascending=False
)

category_funnel

소분류 컬럼

In [ ]:
df['sub_category'] = (
    df['category_code']
    .str.split('.')
    .str[:2]
    .str.join('.')
)

In [ ]:
event_first_time_sub = (
    df
    .groupby([
        'sub_category',
        'user_session',
        'product_id',
        'event_type'
    ])['event_time']
    .min()
    .unstack()
    .reset_index()
)

In [ ]:
for col in ['view', 'cart', 'purchase']:
    if col not in event_first_time_sub.columns:
        event_first_time_sub[col] = pd.NaT

# 존재 여부
event_first_time_sub['has_view'] = (
    event_first_time_sub['view'].notna()
)

event_first_time_sub['has_cart'] = (
    event_first_time_sub['cart'].notna()
)

event_first_time_sub['has_purchase'] = (
    event_first_time_sub['purchase'].notna()
)

# 순차 퍼널
event_first_time_sub['view_to_cart'] = (
    event_first_time_sub['has_view'] &
    event_first_time_sub['has_cart'] &
    (event_first_time_sub['view'] < event_first_time_sub['cart'])
)

event_first_time_sub['view_to_cart_to_purchase'] = (
    event_first_time_sub['has_view'] &
    event_first_time_sub['has_cart'] &
    event_first_time_sub['has_purchase'] &
    (event_first_time_sub['view'] < event_first_time_sub['cart']) &
    (event_first_time_sub['cart'] < event_first_time_sub['purchase'])
)

In [ ]:
funnel_df_sub = event_first_time_sub[[
    'sub_category',
    'user_session',
    'product_id',
    'has_view',
    'view_to_cart',
    'view_to_cart_to_purchase'
]].rename(columns={
    'has_view': 'view'
})

In [ ]:
sub_category_funnel_summary = (
    funnel_df_sub.groupby('sub_category')
    .agg(
        view=('view', 'sum'),
        view_to_cart=('view_to_cart', 'sum'),
        view_to_cart_to_purchase=('view_to_cart_to_purchase', 'sum')
    )
    .reset_index()
)

In [ ]:
sub_category_funnel_summary['view_to_cart_rate'] = (
    sub_category_funnel_summary['view_to_cart'] /
    sub_category_funnel_summary['view'] * 100
)

sub_category_funnel_summary['cart_to_purchase_rate'] = (
    sub_category_funnel_summary['view_to_cart_to_purchase'] /
    sub_category_funnel_summary['view_to_cart'] * 100
)

sub_category_funnel_summary['total_conversion_rate'] = (
    sub_category_funnel_summary['view_to_cart_to_purchase'] /
    sub_category_funnel_summary['view'] * 100
)

In [ ]:
sub_category_funnel_summary = (
    sub_category_funnel_summary
    .sort_values(
        'total_conversion_rate',
        ascending=False
    )
)

sub_category_funnel_summary